# W&B Naming And Figure Logging Verification

Scratch check for the current tracking contract. This notebook uses a fake in-memory `wandb` module, so it does not log online and does not require credentials.

It verifies:
- default W&B run names include `prediction_loss`, such as `eps_eps`;
- W&B config has flat `param` values, such as `epseps` and `vx`;
- W&B tags include `param:<value>`;
- figure logging uses the expected keys and training-step captions.

In [ ]:
from pathlib import Path
import sys
import tempfile
import types

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
assert (repo_root / "src").exists(), f"Could not find repo root from {Path.cwd()}"

sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))

from mp_artifacts import create_run_layout, load_json
from mp_config import clone_config, get_base_config
from mp_tracking import AttemptTracker

repo_root

In [ ]:
class FakeWandbRun:
    def __init__(self, run_id, name):
        self.id = run_id
        self.name = name
        self.path = ["entity", "project", run_id]
        self.url = f"https://wandb.local/{run_id}"
        self.logs = []
        self.finished = False

    def log(self, payload, step=None):
        self.logs.append((payload, step))

    def finish(self):
        self.finished = True


fake_wandb = types.SimpleNamespace(init_calls=[], runs=[])

def fake_init(**kwargs):
    fake_wandb.init_calls.append(kwargs)
    run = FakeWandbRun(kwargs["id"], kwargs["name"])
    fake_wandb.runs.append(run)
    return run

fake_wandb.init = fake_init
fake_wandb.Image = lambda path, caption=None: {
    "image_path": path,
    "caption": caption,
}
fake_wandb.define_metric = lambda *args, **kwargs: None

old_wandb = sys.modules.get("wandb")
sys.modules["wandb"] = fake_wandb

In [ ]:
def make_config(prediction_space="x", loss_space="x"):
    config = clone_config(get_base_config())
    config.study.name = "tracking_demo"
    config.outputs.run_name = "tracking_demo"
    config.training.seed = 3
    config.framework.prediction_space = prediction_space
    config.framework.loss_space = loss_space
    config.tracking.console.enabled = False
    config.tracking.wandb.enabled = True
    config.tracking.wandb.mode = "offline"
    return config


def run_fake_attempt(prediction_space, loss_space):
    config = make_config(prediction_space, loss_space)
    with tempfile.TemporaryDirectory() as tmpdir:
        experiment_dir = Path(tmpdir) / "tracking_demo"
        attempt_dir = create_run_layout(experiment_dir, config)
        initial_status = load_json(attempt_dir / "status.json")
        expected_wandb_id = (
            f"{initial_status['config_hash']}-"
            f"{initial_status['seed_id']}-"
            f"{initial_status['attempt_id']}"
        )

        figure_path = attempt_dir / "figures" / "demo.png"
        figure_path.parent.mkdir(parents=True, exist_ok=True)
        figure_path.write_bytes(b"fake image")

        tracker = AttemptTracker(config, attempt_dir)
        tracker.log_metrics({"train/loss": 1.25}, step=4)
        tracker.register_figure("demo", figure_path)
        tracker.log_figure_metric("generation_monitor/samples", figure_path, step=9)
        tracker.close()

        return {
            "init": fake_wandb.init_calls[-1],
            "logs": list(fake_wandb.runs[-1].logs),
            "status": load_json(attempt_dir / "status.json"),
            "expected_wandb_id": expected_wandb_id,
            "figure_path": str(figure_path.resolve()),
        }

In [ ]:
cases = {
    ("x", "x"): "xx",
    ("eps", "eps"): "epseps",
    ("v", "v"): "vv",
    ("v", "x"): "vx",
}

results = {}
for (prediction_space, loss_space), expected_param in cases.items():
    result = run_fake_attempt(prediction_space, loss_space)
    init = result["init"]
    expected_name_fragment = f"-{prediction_space}_{loss_space}-"

    assert init["id"] == result["expected_wandb_id"]
    assert expected_name_fragment in init["name"], init["name"]
    assert init["config"]["param"] == expected_param
    assert f"param:{expected_param}" in init["tags"]
    assert init["config"]["framework"]["prediction_space"] == prediction_space
    assert init["config"]["framework"]["loss_space"] == loss_space
    results[f"{prediction_space}/{loss_space}"] = {
        "run_name": init["name"],
        "param": init["config"]["param"],
        "tag_present": f"param:{expected_param}" in init["tags"],
    }

results

In [ ]:
sample = run_fake_attempt("eps", "eps")
logs = sample["logs"]

assert logs[0] == ({"train/loss": 1.25, "train/step": 4}, None)
assert "figures/demo" in logs[1][0]
assert logs[2] == (
    {
        "generation_monitor/samples": {
            "image_path": sample["figure_path"],
            "caption": "generation_monitor | grad_step=9",
        },
        "generation_monitor/step": 9,
    },
    None,
)

{
    "metric_log": logs[0],
    "registered_figure_key": list(logs[1][0].keys())[0],
    "step_figure_log": logs[2],
}

In [ ]:
if old_wandb is None:
    sys.modules.pop("wandb", None)
else:
    sys.modules["wandb"] = old_wandb

"W&B naming, param field/tag, and figure logging checks passed."